In [2]:
import requests, re, unicodedata
import pandas as pd
from bs4 import BeautifulSoup

# ---------- helper functions ----------
def date_time(table_cells):
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    return ''.join([bv for i, bv in enumerate(table_cells.strings) if i % 2 == 0][0:-1])

def landing_status(table_cells):
    return [i for i in table_cells.strings][0]

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        return mass[0:mass.find("kg") + 2]
    return 0

def extract_column_from_header(row):
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    colunm_name = ' '.join(row.contents)
    if not (colunm_name.strip().isdigit()):
        return colunm_name.strip()

def safe(cell):
    # link text if there is a link, otherwise the plain cell text
    if cell.a is not None and cell.a.string:
        return cell.a.string
    return cell.get_text(strip=True)

# ---------- Task 1: request the page ----------
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}
response = requests.get(static_url, headers=headers)
print("Status code:", response.status_code)

# ---------- Task 2: BeautifulSoup object ----------
soup = BeautifulSoup(response.text, 'html.parser')
print(soup.title)

# ---------- Task 3: find all tables ----------
html_tables = soup.find_all('table')
print("Number of tables:", len(html_tables))
first_launch_table = html_tables[2]

# ---------- Task 4: column names ----------
column_names = []
for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)
print(column_names)

# ---------- Task 5: launch_dict ----------
launch_dict = dict.fromkeys(column_names)
launch_dict.pop('Date and time ( )', None)
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []

# ---------- Task 6: parse every launch row ----------
extracted_row = 0
for table in soup.find_all('table', "wikitable plainrowheaders collapsible"):
    for rows in table.find_all("tr"):
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
            else:
                flag = False
        else:
            flag = False
        row = rows.find_all('td')
        if flag and len(row) >= 9:
            extracted_row += 1
            datatimelist = date_time(row[0])
            date = datatimelist[0].strip(',')
            time = datatimelist[1] if len(datatimelist) > 1 else ''
            bv = booster_version(row[1])
            if not bv:
                bv = safe(row[1])
            launch_site = safe(row[2])
            payload = safe(row[3])
            payload_mass = get_mass(row[4])
            orbit = safe(row[5])
            customer = safe(row[6])
            launch_outcome = list(row[7].strings)[0]
            booster_landing = landing_status(row[8])

            launch_dict['Flight No.'].append(flight_number)
            launch_dict['Date'].append(date)
            launch_dict['Time'].append(time)
            launch_dict['Version Booster'].append(bv)
            launch_dict['Launch site'].append(launch_site)
            launch_dict['Payload'].append(payload)
            launch_dict['Payload mass'].append(payload_mass)
            launch_dict['Orbit'].append(orbit)
            launch_dict['Customer'].append(customer)
            launch_dict['Launch outcome'].append(launch_outcome)
            launch_dict['Booster landing'].append(booster_landing)

# ---------- Task 7: DataFrame and export ----------
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})
df.to_csv('spacex_web_scraped.csv', index=False)
print("Rows scraped:", extracted_row)
df.head()

Status code: 200
<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>
Number of tables: 29
['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']
Rows scraped: 121


,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,F9 v1.07B0003.1,Failure,4 June 2010,18:45
1,2,CCAFS,Dragon,0,LEO,NASA,Success,F9 v1.07B0004.1,Failure,8 December 2010,15:43
2,3,CCAFS,Dragon,525 kg,LEO,NASA,Success,F9 v1.07B0005.1,No,22 May 2012,07:44
3,4,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,F9 v1.07B0006.1,No attempt,8 October 2012,00:35
4,5,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,F9 v1.07B0007.1,No,1 March 2013,15:10
